# AI Guru Legal RAG — Article-level Doc-first v2 GPU

Bản này là cú thử mạnh nhất sau các kết quả trước:

- Không boost hậu kỳ trên submission.
- Không tăng recall đại trà.
- Xây lại tầng chọn **Điều luật** ở cấp article-level.
- Dùng `BAAI/bge-m3` để tạo article index và `BAAI/bge-reranker-v2-m3` để rerank candidate article.
- Có resume, checkpoint, progress log rõ ràng theo ID.

Ý tưởng: doc-first tốt nhất trước đây giữ precision tốt, nhưng chọn Điều còn yếu. Bản này gom chunk thành từng Điều, rerank ở cấp Điều đầy đủ, rồi chọn bằng dynamic budget theo độ phức tạp câu hỏi.

In [1]:
# ===== 0. Install dependencies =====
!pip install -q faiss-cpu rank-bm25 sentence-transformers

import os, sys, json, time, zipfile, shutil, pickle, re, math, subprocess, textwrap
from pathlib import Path
print("Ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [2]:
# ===== 1. Config =====
from pathlib import Path

WORK_DIR = Path("/kaggle/working/article_level_doc_first_v2")
ART_DIR = WORK_DIR / "artifacts"
OUT_DIR = WORK_DIR / "outputs"
SCRIPT_PATH = WORK_DIR / "article_level_infer.py"

WORK_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Models: giữ model mạnh nhất đã dùng ổn định trước đó
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# Inference settings
DENSE_TOP_K = 90
BM25_TOP_K = 90
PREFILTER_TOP_K = 64
RERANK_TOP_K = 48
MAX_SUBQUERIES = 4
SAVE_EVERY = 2

print("WORK_DIR:", WORK_DIR)
print("ART_DIR:", ART_DIR)
print("OUT_DIR:", OUT_DIR)


WORK_DIR: /kaggle/working/article_level_doc_first_v2
ART_DIR: /kaggle/working/article_level_doc_first_v2/artifacts
OUT_DIR: /kaggle/working/article_level_doc_first_v2/outputs


In [3]:
# ===== 2. Find inputs =====
import zipfile, shutil, json, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def unzip_all_relevant():
    # Kaggle datasets sometimes contain a single zip file. Extract zips once.
    extract_root = Path("/kaggle/working/extracted_inputs")
    extract_root.mkdir(parents=True, exist_ok=True)
    for z in INPUT_ROOT.rglob("*.zip"):
        target = extract_root / z.stem
        marker = target / ".unzipped"
        if marker.exists():
            continue
        print("Unzipping:", z, "->", target)
        target.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(z, "r") as f:
                f.extractall(target)
            marker.write_text("ok")
        except Exception as e:
            print("Skip unzip failed:", z, repr(e))
    return extract_root

EXTRACT_ROOT = unzip_all_relevant()
SEARCH_ROOTS = [Path("/kaggle/input"), EXTRACT_ROOT, Path("/kaggle/working")]

def find_file(patterns):
    if isinstance(patterns, str): patterns = [patterns]
    hits = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pat in patterns:
            hits.extend(root.rglob(pat))
    # Prefer v4 / ready / working artifacts, avoid hidden/system duplicated if possible
    hits = sorted(set(hits), key=lambda p: ("v4" not in str(p).lower(), len(str(p))))
    return hits[0] if hits else None

chunks_pkl = find_file("chunks.pkl")
chunks_jsonl = find_file("legal_chunks_final.jsonl")
test_file = find_file("R2AIStage1DATA.json")

print("chunks.pkl:", chunks_pkl)
print("legal_chunks_final.jsonl:", chunks_jsonl)
print("test_file:", test_file)
assert chunks_pkl or chunks_jsonl, "Không tìm thấy chunks.pkl hoặc legal_chunks_final.jsonl"
assert test_file, "Không tìm thấy R2AIStage1DATA.json"


chunks.pkl: /kaggle/input/datasets/minhhoang0110/ai-guru-v4-ready-artifacts-and-chunks/artifacts_v4_doc_first/chunks.pkl
legal_chunks_final.jsonl: /kaggle/input/datasets/ptlocnguyen/r2ai-legal-chunk-data-v3-test-public/chunk_v4/legal_chunks_final.jsonl
test_file: /kaggle/input/datasets/ptlocnguyen/r2ai-legal-chunk-data-v3-test-public/R2AIStage1DATA.json


In [4]:
# ===== 3. Build article-level artifacts =====
# Chạy một lần. Nếu artifact đã có thì skip.
import os, json, pickle, re, unicodedata, math, time
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

ARTICLE_RECORDS_PKL = ART_DIR / "article_records.pkl"
ARTICLE_RECORDS_JSONL = ART_DIR / "article_records.jsonl"
ARTICLE_BM25_PKL = ART_DIR / "article_bm25.pkl"
ARTICLE_TOKENS_PKL = ART_DIR / "article_tokens.pkl"
ARTICLE_FAISS = ART_DIR / "article_faiss.index"
ARTICLE_EMB = ART_DIR / "article_embeddings.npy"

TYPE_DISPLAY = {
    "LUẬT": "Luật",
    "BỘ LUẬT": "Bộ luật",
    "NGHỊ ĐỊNH": "Nghị định",
    "THÔNG TƯ": "Thông tư",
    "PHÁP LỆNH": "Pháp lệnh",
    "QUYẾT ĐỊNH": "Quyết định",
}

def norm_text(s):
    s = str(s or "").lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = s.replace("đ", "d")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def tokenize(s):
    s = norm_text(s)
    toks = s.split()
    stop = set("va cua la co cho khi neu thi de duoc trong voi ve cac nhung mot hai ba bon nam sau tren duoi theo nay do vao ra hoac dong thoi phai can cong ty doanh nghiep".split())
    return [t for t in toks if len(t) > 1 and t not in stop]

def load_chunks():
    if chunks_pkl and Path(chunks_pkl).exists():
        print("Loading chunks.pkl", chunks_pkl)
        with open(chunks_pkl, "rb") as f:
            return pickle.load(f)
    print("Loading jsonl", chunks_jsonl)
    out=[]
    with open(chunks_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                out.append(json.loads(line))
    return out

def doc_ref_from_meta(m):
    num = str(m.get("canonical_document_number") or m.get("document_number") or "").strip()
    typ = str(m.get("document_type") or "").strip().upper()
    typ_disp = TYPE_DISPLAY.get(typ, typ.title() if typ else "Văn bản")
    title = str(m.get("document_title") or "").strip()
    # Keep uppercase titles from corpus as-is; do not use noisy document_label.
    name = f"{typ_disp} {num} {title}".strip()
    return f"{num}|{name}" if num else name

def article_num_clean(x):
    x = str(x or "").strip()
    m = re.search(r"\d+[a-zA-Z]?", x)
    return m.group(0) if m else x

def article_int(x):
    m = re.search(r"\d+", str(x or ""))
    return int(m.group(0)) if m else 9999

if ARTICLE_RECORDS_PKL.exists() and ARTICLE_FAISS.exists() and ARTICLE_BM25_PKL.exists():
    print("Article-level artifacts already exist, skip build.")
else:
    chunks = load_chunks()
    print("chunks:", len(chunks))
    groups = defaultdict(list)
    meta_best = {}
    for c in chunks:
        m = c.get("metadata", {}) or {}
        doc_num = str(m.get("canonical_document_number") or m.get("document_number") or "").strip()
        art_num = article_num_clean(m.get("canonical_article_number") or m.get("article_number") or "")
        if not doc_num or not art_num:
            continue
        key = f"{doc_num}|{art_num}"
        groups[key].append(c)
        meta_best[key] = m

    records=[]
    for key, cs in groups.items():
        m = meta_best[key]
        doc_ref = doc_ref_from_meta(m)
        doc_num = str(m.get("canonical_document_number") or m.get("document_number") or "").strip()
        art_num = article_num_clean(m.get("canonical_article_number") or m.get("article_number") or "")
        article_title = str(m.get("article_title") or "").strip()
        article_label = str(m.get("article_label") or f"Điều {art_num}. {article_title}").strip()
        # Sort chunks in article. Use content, fallback to chunk_text.
        cs_sorted = sorted(cs, key=lambda x: int(x.get("chunk_index_in_article") or 0))
        contents=[]
        for c in cs_sorted:
            txt = str(c.get("content") or c.get("chunk_text") or "").strip()
            if txt and txt not in contents:
                contents.append(txt)
        full_content = "\n\n".join(contents)
        if not full_content:
            continue
        article_ref = f"{doc_ref}|Điều {art_num}"
        title_blob = " | ".join([doc_num, str(m.get("document_title") or ""), article_label, article_title, str(m.get("category_name") or "")])
        retrieval_text = f"Văn bản: {doc_ref}\nĐiều: Điều {art_num}. {article_title}\nTừ khóa: {title_blob}\nNội dung:\n{full_content}"
        records.append({
            "article_key": key,
            "doc_ref": doc_ref,
            "article_ref": article_ref,
            "doc_number": doc_num,
            "document_title": str(m.get("document_title") or ""),
            "document_type": str(m.get("document_type") or ""),
            "category_name": str(m.get("category_name") or ""),
            "article_number": art_num,
            "article_number_int": article_int(art_num),
            "article_title": article_title,
            "article_label": article_label,
            "content": full_content,
            "retrieval_text": retrieval_text,
            "title_text": title_blob,
            "tokens": tokenize(retrieval_text),
            "title_tokens": tokenize(title_blob),
        })
    records.sort(key=lambda r: (r["doc_number"], r["article_number_int"], r["article_ref"]))
    print("article records:", len(records))
    print("sample:", records[0]["article_ref"])

    with open(ARTICLE_RECORDS_PKL, "wb") as f:
        pickle.dump(records, f)
    with open(ARTICLE_RECORDS_JSONL, "w", encoding="utf-8") as f:
        for r in records:
            rr = dict(r); rr.pop("tokens", None); rr.pop("title_tokens", None)
            f.write(json.dumps(rr, ensure_ascii=False) + "\n")

    from rank_bm25 import BM25Okapi
    tokens = [r["tokens"] for r in records]
    bm25 = BM25Okapi(tokens)
    with open(ARTICLE_BM25_PKL, "wb") as f:
        pickle.dump(bm25, f)
    with open(ARTICLE_TOKENS_PKL, "wb") as f:
        pickle.dump(tokens, f)
    print("BM25 built")

    import torch, faiss
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Embedding device:", device)
    model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
    texts = [r["retrieval_text"][:3600] for r in records]
    emb = model.encode(texts, batch_size=24 if device=="cuda" else 8, normalize_embeddings=True, show_progress_bar=True)
    emb = np.asarray(emb, dtype="float32")
    np.save(ARTICLE_EMB, emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, str(ARTICLE_FAISS))
    print("FAISS built:", ARTICLE_FAISS, "dim", emb.shape)

print("Artifacts ready:")
for p in [ARTICLE_RECORDS_PKL, ARTICLE_BM25_PKL, ARTICLE_FAISS, ARTICLE_EMB]:
    print(p, p.exists(), round(p.stat().st_size/1024/1024,2) if p.exists() else None, "MB")


Loading chunks.pkl /kaggle/input/datasets/minhhoang0110/ai-guru-v4-ready-artifacts-and-chunks/artifacts_v4_doc_first/chunks.pkl
chunks: 12299
article records: 10785
sample: 01/2016/QH14|Luật 01/2016/QH14 ĐẤU GIÁ TÀI SẢN|Điều 1
BM25 built


Embedding device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/450 [00:00<?, ?it/s]

FAISS built: /kaggle/working/article_level_doc_first_v2/artifacts/article_faiss.index dim (10785, 1024)
Artifacts ready:
/kaggle/working/article_level_doc_first_v2/artifacts/article_records.pkl True 89.04 MB
/kaggle/working/article_level_doc_first_v2/artifacts/article_bm25.pkl True 8.05 MB
/kaggle/working/article_level_doc_first_v2/artifacts/article_faiss.index True 42.13 MB
/kaggle/working/article_level_doc_first_v2/artifacts/article_embeddings.npy True 42.13 MB


In [5]:
# ===== 4. Write inference script =====
SCRIPT = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Article-level Doc-first v2 inference.
- Builds no post-hoc manual fixes: all decisions happen inside this pipeline.
- Resume-safe, frequent checkpoints, explicit progress by ID.
"""
import os, sys, json, pickle, re, unicodedata, time, argparse, zipfile, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np


def norm_text(s):
    s = str(s or "").lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = s.replace("đ", "d")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

STOP = set("va cua la co cho khi neu thi de duoc trong voi ve cac nhung mot hai ba bon nam sau tren duoi theo nay do vao ra hoac dong thoi phai can cong ty doanh nghiep nguoi ben viec noi dung quy dinh truong hop".split())
def tokenize(s):
    toks = norm_text(s).split()
    return [t for t in toks if len(t)>1 and t not in STOP]

DOMAIN_RULES = [
    ("sme_support", ["doanh nghiep nho va vua", "dnvv", "dnnvv", "ho tro doanh nghiep", "sieu nho", "khoi nghiep sang tao", "chuoi gia tri", "cum lien ket", "tu van vien", "dao tao", "mat bang san xuat"],
     ["04/2017/QH14", "80/2021/NĐ-CP", "06/2022/TT-BKHDT", "52/2023/TT-BTC", "39/2019/NĐ-CP", "34/2018/NĐ-CP", "38/2018/NĐ-CP"]),
    ("startup_fund", ["quy dau tu khoi nghiep", "quy dau tu khoi nghiep sang tao", "dau tu khoi nghiep sang tao", "cung dau tu"],
     ["38/2018/NĐ-CP", "04/2017/QH14", "80/2021/NĐ-CP"]),
    ("business_reg", ["dang ky doanh nghiep", "dang ky kinh doanh", "thanh lap cong ty", "ten doanh nghiep", "tru so chinh", "chi nhanh", "giai the", "tam ngung", "ho kinh doanh", "giay chung nhan dang ky"],
     ["168/2025/NĐ-CP", "59/2020/QH14"]),
    ("corporate", ["cong ty co phan", "dai hoi dong co dong", "hoi dong thanh vien", "von dieu le", "gop von", "chuyen nhuong von", "thanh vien moi", "nguoi dai dien"],
     ["59/2020/QH14", "168/2025/NĐ-CP"]),
    ("tax", ["thue", "khai thue", "quan ly thue", "mien thue", "giam thue", "hoan thue", "thu nhap doanh nghiep", "tieu thu dac biet", "nop thue", "xu phat thue"],
     ["38/2019/QH14", "80/2021/TT-BTC", "126/2020/NĐ-CP", "125/2020/NĐ-CP", "67/2025/QH15", "320/2025/NĐ-CP"]),
    ("labor", ["lao dong", "nhan vien", "hop dong lao dong", "thu viec", "tien luong", "bao hiem xa hoi", "tai nan lao dong", "benh nghe nghiep", "an toan ve sinh lao dong", "cong doan", "dinh cong", "thoa uoc lao dong"],
     ["45/2019/QH14", "12/2022/NĐ-CP", "145/2020/NĐ-CP", "84/2015/QH13", "28/2021/TT-BLĐTBXH", "50/2024/QH15"]),
    ("accounting", ["ke toan", "bao cao tai chinh", "kiem toan", "chung tu ke toan", "so ke toan", "doanh nghiep sieu nho"],
     ["88/2015/QH13", "133/2016/TT-BTC", "132/2018/TT-BTC", "41/2018/NĐ-CP"]),
    ("accounting_penalty", ["xu phat ke toan", "vi pham ke toan", "phat ke toan", "che tai ke toan", "xu phat kiem toan", "vi pham kiem toan"],
     ["41/2018/NĐ-CP", "88/2015/QH13"]),
    ("commercial", ["thuong mai", "mua ban hang hoa", "dai ly thuong mai", "logistics", "nhuong quyen thuong mai", "khuyen mai", "hoi cho", "trien lam", "van chuyen hang hoa", "quang cao thuong mai"],
     ["36/2005/QH11"]),
    ("civil", ["dan su", "giao dich dan su", "hop dong", "vo hieu", "nang luc hanh vi", "tai san bao dam", "cam co", "the chap", "bao lanh", "xu ly tai san"],
     ["91/2015/QH13", "21/2021/NĐ-CP"]),
    ("arbitration", ["trong tai", "thoa thuan trong tai", "ban trong tai", "hoi dong trong tai", "to tung trong tai"],
     ["54/2010/QH12"]),
    ("consumer", ["nguoi tieu dung", "khach hang", "hop dong theo mau", "dieu kien giao dich chung", "bao ve thong tin", "du lieu khach hang", "xoa du lieu", "mien trach nhiem"],
     ["19/2023/QH15", "55/2024/NĐ-CP"]),
    ("ip", ["so huu tri tue", "sang che", "nhan hieu", "chi dan dia ly", "thanh dinh noi dung", "don dang ky"],
     ["50/2005/QH11"]),
]

GENERIC_TITLES = ["pham vi", "doi tuong ap dung", "giai thich tu ngu", "nguyen tac", "chinh sach", "trach nhiem", "hieu luc", "dieu khoan thi hanh"]
ALLOW_GENERIC_Q = ["pham vi", "doi tuong", "giai thich", "khai niem", "nguyen tac", "tieu chi", "dieu kien", "trach nhiem", "hieu luc", "nguon von", "can cu"]


def detect_domains(q):
    nq = norm_text(q)
    docs = set(); names=set()
    for name, kws, docnums in DOMAIN_RULES:
        if any(kw in nq for kw in kws):
            names.add(name); docs.update(docnums)
    return names, docs


def split_subqueries(q, max_sub=4):
    q = str(q or "").strip()
    parts = [q]
    # Split multi-issue legal questions but keep meaningful clauses.
    seps = [" đồng thời ", ";", " và nếu ", " nếu ", " trong trường hợp ", " thì ", " và "]
    tmp=[q]
    for sep in seps:
        new=[]
        for x in tmp:
            new += [p.strip(" ,.;:-") for p in x.split(sep) if len(p.strip())>=25]
        tmp = new if len(new)>1 else tmp
    # Prefer clauses with legal-action words.
    scored=[]
    for p in [q]+tmp:
        np_ = norm_text(p)
        if len(np_)<18: continue
        score = len(np_.split())
        for kw in ["xu phat", "khac phuc", "ho so", "dieu kien", "nghia vu", "quyen", "thoi han", "thu tuc", "dang ky", "mien giam", "trach nhiem"]:
            if kw in np_: score += 10
        scored.append((score,p))
    out=[]; seen=set()
    for _,p in sorted(scored, reverse=True):
        k=norm_text(p)
        if k not in seen:
            out.append(p); seen.add(k)
        if len(out)>=max_sub: break
    return out or [q]


def complexity(q):
    nq = norm_text(q)
    c = 0
    for kw in ["dong thoi", "va neu", "nhung", "cac", "bao gom", "nhu the nao va", "quy trinh", "ho so", "nghia vu", "quyen", "xu ly", "khac phuc", "muc xu phat", "thoi han"]:
        if kw in nq: c += 1
    # Multiple legal domains often require multiple articles.
    domains,_ = detect_domains(q)
    if len(domains) >= 2: c += 1
    return c


def title_overlap_score(q_tokens, title_tokens):
    if not q_tokens or not title_tokens:
        return 0.0
    qs=set(q_tokens); ts=set(title_tokens)
    inter=len(qs & ts)
    return min(1.0, inter / max(3, min(len(qs), len(ts))))


def is_generic_article(rec, q):
    art_int = rec.get("article_number_int", 9999)
    title = norm_text(rec.get("article_title", "") + " " + rec.get("article_label", ""))
    nq = norm_text(q)
    if any(k in nq for k in ALLOW_GENERIC_Q):
        return False
    if art_int <= 5 and any(k in title for k in GENERIC_TITLES):
        return True
    return False


def answer_template(question, selected):
    refs = [r["article_ref"] for r in selected]
    lines = ["Dựa trên các căn cứ pháp lý truy xuất được, cần đối chiếu trực tiếp các quy định dưới đây để xác định quyền, nghĩa vụ, điều kiện hoặc chế tài áp dụng cho tình huống được hỏi.", "", "Căn cứ pháp lý:"]
    for ref in refs:
        lines.append(f"- {ref}")
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--test_file", required=True)
    ap.add_argument("--artifact_dir", required=True)
    ap.add_argument("--output_file", required=True)
    ap.add_argument("--debug_file", required=True)
    ap.add_argument("--embedding_model_name", default="BAAI/bge-m3")
    ap.add_argument("--reranker_model_name", default="BAAI/bge-reranker-v2-m3")
    ap.add_argument("--offset", type=int, default=0)
    ap.add_argument("--limit", type=int, default=1000)
    ap.add_argument("--resume", type=int, default=1)
    ap.add_argument("--save_every", type=int, default=2)
    ap.add_argument("--dense_top_k", type=int, default=90)
    ap.add_argument("--bm25_top_k", type=int, default=90)
    ap.add_argument("--prefilter_top_k", type=int, default=64)
    ap.add_argument("--rerank_top_k", type=int, default=48)
    ap.add_argument("--max_subqueries", type=int, default=4)
    args = ap.parse_args()

    import torch, faiss
    from sentence_transformers import SentenceTransformer, CrossEncoder

    art_dir = Path(args.artifact_dir)
    with open(art_dir/"article_records.pkl", "rb") as f:
        records = pickle.load(f)
    with open(art_dir/"article_bm25.pkl", "rb") as f:
        bm25 = pickle.load(f)
    index = faiss.read_index(str(art_dir/"article_faiss.index"))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[init] device={device} records={len(records)} offset={args.offset} limit={args.limit}", flush=True)
    emb_model = SentenceTransformer(args.embedding_model_name, device=device)
    reranker = CrossEncoder(args.reranker_model_name, device=device, max_length=512)

    with open(args.test_file, "r", encoding="utf-8") as f:
        test = json.load(f)
    subset = test[args.offset: args.offset + args.limit]

    out_path=Path(args.output_file); dbg_path=Path(args.debug_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    outputs=[]; debugs=[]; done_ids=set()
    if args.resume and out_path.exists():
        try:
            outputs=json.load(open(out_path,"r",encoding="utf-8"))
            done_ids={int(x["id"]) for x in outputs}
            print(f"[resume] loaded outputs={len(outputs)} last_id={max(done_ids) if done_ids else None}", flush=True)
        except Exception as e:
            print("[resume] failed output",repr(e),flush=True)
    if args.resume and dbg_path.exists():
        try:
            debugs=json.load(open(dbg_path,"r",encoding="utf-8"))
        except Exception:
            debugs=[]

    def save():
        outputs_sorted=sorted(outputs,key=lambda x:int(x["id"]))
        json.dump(outputs_sorted, open(out_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
        json.dump(debugs, open(dbg_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

    def retrieve_candidates(question):
        subqs = split_subqueries(question, args.max_subqueries)
        q_tokens = tokenize(question)
        domain_names, domain_docs = detect_domains(question)
        cand = {}
        def add(idx, dense=0.0, bm=0.0, source=""):
            if idx < 0 or idx >= len(records): return
            d = cand.setdefault(idx, {"dense":0.0,"bm25":0.0,"sources":set()})
            d["dense"] = max(d["dense"], float(dense))
            d["bm25"] = max(d["bm25"], float(bm))
            if source: d["sources"].add(source)

        for qi, sq in enumerate(subqs):
            q_emb = emb_model.encode([sq], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
            D,I = index.search(q_emb, min(args.dense_top_k, len(records)))
            for rank,(idx,score) in enumerate(zip(I[0],D[0])):
                add(int(idx), dense=float(score), source=f"dense{qi}")
            toks=tokenize(sq)
            scores=bm25.get_scores(toks)
            if len(scores):
                topn=min(args.bm25_top_k, len(scores))
                top_idx=np.argpartition(scores, -topn)[-topn:]
                maxs=float(scores[top_idx].max()) if top_idx.size else 1.0
                if maxs <= 0: maxs=1.0
                for idx in top_idx:
                    add(int(idx), bm=float(scores[idx]/maxs), source=f"bm25{qi}")

        # prefilter score, cheap features
        scored=[]
        for idx,d in cand.items():
            r=records[idx]
            ttl=title_overlap_score(q_tokens, r.get("title_tokens") or tokenize(r.get("title_text","")))
            doc_bonus=0.0
            if r.get("doc_number") in domain_docs:
                doc_bonus=1.0
            # do not hard-penalize cross-domain; multi-domain questions exist.
            generic_pen = 0.10 if is_generic_article(r, question) else 0.0
            pre = 0.45*d["dense"] + 0.35*d["bm25"] + 0.12*ttl + 0.08*doc_bonus - generic_pen
            scored.append((pre, idx, ttl, doc_bonus, generic_pen, d))
        scored.sort(reverse=True, key=lambda x:x[0])
        return scored[:args.prefilter_top_k], subqs, sorted(domain_names), sorted(domain_docs)

    def infer_one(item):
        q = item.get("question") or item.get("query") or item.get("prompt") or ""
        candidates, subqs, domain_names, domain_docs = retrieve_candidates(q)
        if not candidates:
            return {"id":item["id"],"question":q,"answer":"Không tìm thấy căn cứ pháp lý phù hợp.","relevant_docs":[],"relevant_articles":[]}, {"id":item["id"],"error":"no candidates"}

        rerank_items = candidates[:args.rerank_top_k]
        pairs=[(q, records[idx]["retrieval_text"][:3000]) for _,idx,_,_,_,_ in rerank_items]
        try:
            rr=np.asarray(reranker.predict(pairs, batch_size=16, show_progress_bar=False), dtype="float32")
        except TypeError:
            rr=np.asarray(reranker.predict(pairs, batch_size=16), dtype="float32")
        if len(rr)>0:
            rr_min=float(rr.min()); rr_max=float(rr.max())
            rr_norm=(rr-rr_min)/(rr_max-rr_min+1e-6)
        else:
            rr_norm=rr

        q_tokens=tokenize(q)
        final=[]
        for j,(pre, idx, ttl, doc_bonus, generic_pen, d) in enumerate(rerank_items):
            r=records[idx]
            dense=float(d["dense"]); bm=float(d["bm25"])
            rrs=float(rr_norm[j]) if j < len(rr_norm) else 0.0
            # More weight on reranker but preserve lexical/title/domain evidence.
            score=0.58*rrs + 0.18*dense + 0.12*bm + 0.08*ttl + 0.06*doc_bonus - generic_pen
            final.append({"idx":idx,"rec":r,"score":score,"rerank":float(rr[j]) if j<len(rr) else None,"rerank_norm":rrs,"dense":dense,"bm25":bm,"title":ttl,"doc_bonus":doc_bonus,"generic_pen":generic_pen,"sources":list(d["sources"])})
        final.sort(key=lambda x:x["score"], reverse=True)

        comp=complexity(q)
        nq=norm_text(q)
        if comp <= 0:
            max_articles=2; max_docs=2; rel=0.84; per_doc=2
        elif comp == 1:
            max_articles=3; max_docs=3; rel=0.78; per_doc=2
        elif comp == 2:
            max_articles=4; max_docs=3; rel=0.73; per_doc=2
        else:
            max_articles=5; max_docs=4; rel=0.68; per_doc=3
        # Some broad support questions legitimately need 4 articles, but avoid 6+.
        if any(k in nq for k in ["nhung noi dung", "bao gom nhung", "nhung chi phi", "nhung loai", "nhung hinh thuc"]):
            max_articles=max(max_articles,4); max_docs=max(max_docs,3); rel=min(rel,0.74)

        selected=[]; seen_art=set(); doc_counts=Counter(); selected_docs=[]; seen_docs=set()
        top_score=final[0]["score"] if final else 0
        for it in final:
            r=it["rec"]
            if r["article_ref"] in seen_art:
                continue
            if len(selected) >= max_articles:
                break
            if r["doc_ref"] not in seen_docs and len(seen_docs) >= max_docs:
                continue
            if doc_counts[r["doc_ref"]] >= per_doc:
                continue
            # Relative score gate. Always take top1, then score-based additions.
            if selected and it["score"] < top_score * rel:
                # Allow very strong domain/title matches for complex questions only.
                if not (comp >= 2 and it["doc_bonus"] > 0 and it["title"] >= 0.25 and it["score"] >= top_score*0.62):
                    continue
            # If generic and already have a specific article from same doc, skip unless it is high.
            if is_generic_article(r, q) and doc_counts[r["doc_ref"]] > 0 and it["score"] < top_score*0.90:
                continue
            selected.append(r)
            seen_art.add(r["article_ref"])
            doc_counts[r["doc_ref"]] += 1
            if r["doc_ref"] not in seen_docs:
                seen_docs.add(r["doc_ref"]); selected_docs.append(r["doc_ref"])

        # Minimal fallback: if only one article for complex question, add next high-confidence different aspect.
        if comp >= 2 and len(selected) < 2:
            for it in final:
                r=it["rec"]
                if r["article_ref"] in seen_art: continue
                if len(selected) >= min(3,max_articles): break
                if r["doc_ref"] not in seen_docs and len(seen_docs) >= max_docs: continue
                if it["score"] >= top_score*0.60:
                    selected.append(r); seen_art.add(r["article_ref"]); doc_counts[r["doc_ref"]]+=1
                    if r["doc_ref"] not in seen_docs:
                        seen_docs.add(r["doc_ref"]); selected_docs.append(r["doc_ref"])

        relevant_articles=[r["article_ref"] for r in selected]
        relevant_docs=[]; sd=set()
        for r in selected:
            if r["doc_ref"] not in sd:
                relevant_docs.append(r["doc_ref"]); sd.add(r["doc_ref"])
        ans=answer_template(q, selected)
        out={"id":item["id"],"question":q,"answer":ans,"relevant_docs":relevant_docs,"relevant_articles":relevant_articles}
        dbg={
            "id":item["id"],"question":q,"complexity":comp,"subqueries":subqs,"domains":domain_names,"domain_docs":domain_docs,
            "selected":[{"article_ref":r["article_ref"],"doc_ref":r["doc_ref"],"article_title":r.get("article_title") } for r in selected],
            "top_candidates":[{"ref":it["rec"]["article_ref"],"score":round(it["score"],4),"rr":round(it["rerank_norm"],4),"dense":round(it["dense"],4),"bm25":round(it["bm25"],4),"title":round(it["title"],4),"doc_bonus":it["doc_bonus"],"generic_pen":it["generic_pen"]} for it in final[:10]],
        }
        return out, dbg

    start=time.time(); processed=0
    total=len(subset)
    for local_i,item in enumerate(subset, start=1):
        iid=int(item["id"])
        if iid in done_ids:
            continue
        try:
            out,dbg=infer_one(item)
        except Exception as e:
            print(f"[error] id={iid} {repr(e)}", flush=True)
            out={"id":iid,"question":item.get("question",""),"answer":"Không tìm thấy căn cứ pháp lý phù hợp.","relevant_docs":[],"relevant_articles":[]}
            dbg={"id":iid,"error":repr(e)}
        outputs.append(out); debugs.append(dbg); done_ids.add(iid); processed+=1
        if processed % 1 == 0:
            elapsed=time.time()-start
            print(f"[progress] offset={args.offset} done_new={processed} total_done={len(done_ids)}/{total} id={iid} elapsed={elapsed/60:.1f}m arts={len(out['relevant_articles'])} docs={len(out['relevant_docs'])}", flush=True)
        if processed % args.save_every == 0:
            save()
    save()
    print(f"[finished] offset={args.offset} outputs={len(outputs)} elapsed={(time.time()-start)/60:.1f}m", flush=True)

if __name__ == "__main__":
    main()
'''
SCRIPT_PATH.write_text(SCRIPT, encoding='utf-8')
print('Wrote', SCRIPT_PATH)


Wrote /kaggle/working/article_level_doc_first_v2/article_level_infer.py


In [6]:
# ===== 5. Run two-GPU inference =====
# Cell này chạy 2 process song song nếu có T4x2. Có thể interrupt an toàn, sau đó rerun với RESUME=1.
import subprocess, os, time, textwrap
from pathlib import Path

RESUME = 1
LIMIT0 = 1000
LIMIT1 = 1000

part0_out = OUT_DIR / "submission_part0_article_v2.json"
part0_dbg = OUT_DIR / "submission_part0_article_v2_debug.json"
part1_out = OUT_DIR / "submission_part1_article_v2.json"
part1_dbg = OUT_DIR / "submission_part1_article_v2_debug.json"
log0 = OUT_DIR / "part0.log"
log1 = OUT_DIR / "part1.log"

cmd0 = f"""
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python {SCRIPT_PATH} \
  --test_file {test_file} \
  --artifact_dir {ART_DIR} \
  --output_file {part0_out} \
  --debug_file {part0_dbg} \
  --embedding_model_name {EMBEDDING_MODEL_NAME} \
  --reranker_model_name {RERANKER_MODEL_NAME} \
  --offset 0 --limit {LIMIT0} --resume {RESUME} --save_every {SAVE_EVERY} \
  --dense_top_k {DENSE_TOP_K} --bm25_top_k {BM25_TOP_K} --prefilter_top_k {PREFILTER_TOP_K} --rerank_top_k {RERANK_TOP_K} --max_subqueries {MAX_SUBQUERIES}
""".strip()

cmd1 = f"""
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=1 python {SCRIPT_PATH} \
  --test_file {test_file} \
  --artifact_dir {ART_DIR} \
  --output_file {part1_out} \
  --debug_file {part1_dbg} \
  --embedding_model_name {EMBEDDING_MODEL_NAME} \
  --reranker_model_name {RERANKER_MODEL_NAME} \
  --offset 1000 --limit {LIMIT1} --resume {RESUME} --save_every {SAVE_EVERY} \
  --dense_top_k {DENSE_TOP_K} --bm25_top_k {BM25_TOP_K} --prefilter_top_k {PREFILTER_TOP_K} --rerank_top_k {RERANK_TOP_K} --max_subqueries {MAX_SUBQUERIES}
""".strip()

print("CMD0:\n", cmd0[:1000], "...\n")
print("CMD1:\n", cmd1[:1000], "...\n")

p0 = subprocess.Popen(cmd0, shell=True, stdout=open(log0, "a"), stderr=subprocess.STDOUT)
p1 = subprocess.Popen(cmd1, shell=True, stdout=open(log1, "a"), stderr=subprocess.STDOUT)
print("Started PIDs:", p0.pid, p1.pid)
print("Logs:", log0, log1)

# Monitor every 60s. Interrupt this cell safely; output files are checkpointed.
try:
    while True:
        r0, r1 = p0.poll(), p1.poll()
        print("\n--- monitor", time.ctime(), "status", r0, r1, "---")
        for name, p, lg in [("part0", part0_out, log0), ("part1", part1_out, log1)]:
            done = 0; rng = None
            if p.exists():
                try:
                    data = json.load(open(p, "r", encoding="utf-8"))
                    done = len(data)
                    if done:
                        ids = sorted(int(x["id"]) for x in data)
                        rng = (ids[0], ids[-1])
                except Exception as e:
                    rng = f"json_err={e}"
            print(name, "done", done, "range", rng, "modified", time.ctime(p.stat().st_mtime) if p.exists() else None)
            if lg.exists():
                tail = lg.read_text(errors="ignore").splitlines()[-3:]
                for line in tail: print(" ", line[-220:])
        if r0 is not None and r1 is not None:
            break
        time.sleep(60)
finally:
    print("Monitor ended. If interrupted, processes may continue in background only until kernel/session stops.")


CMD0:
 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=0 python /kaggle/working/article_level_doc_first_v2/article_level_infer.py   --test_file /kaggle/input/datasets/ptlocnguyen/r2ai-legal-chunk-data-v3-test-public/R2AIStage1DATA.json   --artifact_dir /kaggle/working/article_level_doc_first_v2/artifacts   --output_file /kaggle/working/article_level_doc_first_v2/outputs/submission_part0_article_v2.json   --debug_file /kaggle/working/article_level_doc_first_v2/outputs/submission_part0_article_v2_debug.json   --embedding_model_name BAAI/bge-m3   --reranker_model_name BAAI/bge-reranker-v2-m3   --offset 0 --limit 1000 --resume 1 --save_every 2   --dense_top_k 90 --bm25_top_k 90 --prefilter_top_k 64 --rerank_top_k 48 --max_subqueries 4 ...

CMD1:
 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True CUDA_VISIBLE_DEVICES=1 python /kaggle/working/article_level_doc_first_v2/article_level_infer.py   --test_file /kaggle/input/datasets/ptlocnguyen/r2ai-legal-chunk-data-v3-test-p

In [7]:
# ===== 6. Progress report anytime =====
from pathlib import Path
import json, time, zipfile, os

def progress_report():
    for name, p, lg in [
        ("part0", OUT_DIR / "submission_part0_article_v2.json", OUT_DIR / "part0.log"),
        ("part1", OUT_DIR / "submission_part1_article_v2.json", OUT_DIR / "part1.log"),
    ]:
        print("\n==", name, "==")
        print("file:", p, "exists:", p.exists())
        if p.exists():
            print("size MB:", round(p.stat().st_size/1024/1024,2), "modified:", time.ctime(p.stat().st_mtime))
            try:
                data=json.load(open(p,"r",encoding="utf-8"))
                ids=sorted(int(x["id"]) for x in data)
                print("items:", len(data), "range:", (ids[0],ids[-1]) if ids else None, "last5:", ids[-5:])
                if data:
                    import statistics
                    print("avg docs:", round(statistics.mean(len(x.get('relevant_docs',[])) for x in data),3))
                    print("avg articles:", round(statistics.mean(len(x.get('relevant_articles',[])) for x in data),3))
            except Exception as e:
                print("json error:", repr(e))
        if lg.exists():
            print("log tail:")
            for line in lg.read_text(errors="ignore").splitlines()[-10:]:
                print(line[-250:])

progress_report()



== part0 ==
file: /kaggle/working/article_level_doc_first_v2/outputs/submission_part0_article_v2.json exists: True
size MB: 1.33 modified: Mon Jun 29 15:58:06 2026
items: 1000 range: (1, 1000) last5: [996, 997, 998, 999, 1000]
avg docs: 1.677
avg articles: 2.558
log tail:
[progress] offset=0 done_new=992 total_done=992/1000 id=992 elapsed=87.0m arts=5 docs=2
[progress] offset=0 done_new=993 total_done=993/1000 id=993 elapsed=87.1m arts=3 docs=2
[progress] offset=0 done_new=994 total_done=994/1000 id=994 elapsed=87.2m arts=3 docs=1
[progress] offset=0 done_new=995 total_done=995/1000 id=995 elapsed=87.3m arts=1 docs=1
[progress] offset=0 done_new=996 total_done=996/1000 id=996 elapsed=87.4m arts=1 docs=1
[progress] offset=0 done_new=997 total_done=997/1000 id=997 elapsed=87.5m arts=1 docs=1
[progress] offset=0 done_new=998 total_done=998/1000 id=998 elapsed=87.6m arts=3 docs=2
[progress] offset=0 done_new=999 total_done=999/1000 id=999 elapsed=87.6m arts=3 docs=1
[progress] offset=0 do

In [8]:
# ===== 7. Merge and create submission.zip =====
import json, zipfile, statistics
from pathlib import Path

part0 = OUT_DIR / "submission_part0_article_v2.json"
part1 = OUT_DIR / "submission_part1_article_v2.json"
assert part0.exists(), f"Missing {part0}"
assert part1.exists(), f"Missing {part1}"

data=[]
for p in [part0, part1]:
    arr=json.load(open(p,"r",encoding="utf-8"))
    print(p.name, len(arr))
    data.extend(arr)

# sort, dedup by id keeping latest occurrence
byid={int(x["id"]): x for x in data}
merged=[byid[i] for i in sorted(byid)]
print("merged", len(merged), "id range", merged[0]["id"], merged[-1]["id"])
missing=[i for i in range(1,2001) if i not in byid]
print("missing", len(missing), missing[:20])
assert len(merged)==2000 and not missing, "Chưa đủ 2000 câu; không nên nộp."

# Validate schema only; do not manually alter retrieval decisions.
for x in merged:
    assert "answer" in x and "relevant_docs" in x and "relevant_articles" in x
    # remove duplicates while preserving order inside pipeline packaging; this is schema hygiene, not adding/removing new refs.
    x["relevant_docs"] = list(dict.fromkeys(x.get("relevant_docs", [])))
    x["relevant_articles"] = list(dict.fromkeys(x.get("relevant_articles", [])))

out_json = Path("/kaggle/working/results.json")
json.dump(merged, open(out_json,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
out_zip = Path("/kaggle/working/submission_article_level_doc_first_v2.zip")
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname="results.json")

print("Created:", out_zip)
print("avg docs:", round(statistics.mean(len(x["relevant_docs"]) for x in merged),3))
print("avg articles:", round(statistics.mean(len(x["relevant_articles"]) for x in merged),3))
from collections import Counter
print("article distribution:", Counter(len(x["relevant_articles"]) for x in merged))
print("doc distribution:", Counter(len(x["relevant_docs"]) for x in merged))


submission_part0_article_v2.json 1000
submission_part1_article_v2.json 1000
merged 2000 id range 1 2000
missing 0 []
Created: /kaggle/working/submission_article_level_doc_first_v2.zip
avg docs: 1.726
avg articles: 2.641
article distribution: Counter({2: 739, 3: 471, 1: 320, 4: 279, 5: 191})
doc distribution: Counter({2: 883, 1: 844, 3: 250, 4: 23})


In [9]:
# ===== 8. Pack progress before GPU/session ends =====
# Nếu gần hết quota, interrupt cell chạy, rồi chạy cell này để tải progress và resume sau.
import time, zipfile
from pathlib import Path

def pack_progress():
    ts=time.strftime("%Y%m%d_%H%M%S")
    zpath=Path(f"/kaggle/working/article_level_doc_first_v2_progress_{ts}.zip")
    with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in [OUT_DIR / "submission_part0_article_v2.json", OUT_DIR / "submission_part1_article_v2.json",
                  OUT_DIR / "submission_part0_article_v2_debug.json", OUT_DIR / "submission_part1_article_v2_debug.json",
                  OUT_DIR / "part0.log", OUT_DIR / "part1.log"]:
            if p.exists():
                z.write(p, arcname=str(p.relative_to(WORK_DIR)))
        # article artifacts are useful to avoid rebuilding if needed
        for p in ART_DIR.glob("article_*"):
            if p.exists():
                z.write(p, arcname=str(p.relative_to(WORK_DIR)))
    print("Packed:", zpath)
    return zpath

pack_progress()


Packed: /kaggle/working/article_level_doc_first_v2_progress_20260629_155854.zip


PosixPath('/kaggle/working/article_level_doc_first_v2_progress_20260629_155854.zip')